In [1]:
import os
import json
import time
import random
import signal
import hashlib
from typing import Dict, List, Iterable, Optional, Tuple
import requests

# ========= CONFIG GERAL =========
HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")
MODEL_KEY = os.environ.get("MODEL_KEY", "gemma2:9b")  # ex: "deepseek-r1:14b", etc.
OUTPUT_ROOT = os.environ.get("NER_OUTPUT_ROOT", "./checkpoints")
CACHE_PATH = os.environ.get("NER_CACHE_PATH", "ner_cache.jsonl")

# Pastas por modelo
MODEL_DIR = os.path.join(OUTPUT_ROOT, MODEL_KEY)
os.makedirs(MODEL_DIR, exist_ok=True)

# ======= PROMPTING =======
def format_few_shot_block(examples: List[Dict]) -> str:
    """
    examples: lista de dicts com {"tokens": [...], "tags": [...]}
    """
    lines = []
    for ex in examples:
        toks = " ".join(ex["tokens"])
        tags = " ".join(ex["tags"])
        lines.append(f"INPUT: {toks}\nTAGS: {tags}")
    return "\n\n".join(lines)

def build_prompt(few_shots: List[Dict], tokens: List[str]) -> str:
    few = format_few_shot_block(few_shots) if few_shots else ""
    inp = " ".join(tokens)
    return (
        "Você é um rotulador NER. Dado um texto tokenizado, retorne as TAGS NER por token, "
        "separadas por espaço, usando o mesmo comprimento.\n\n"
        "Exemplos:\n"
        f"{few}\n\n"
        "Agora rotule:\n"
        f"INPUT: {inp}\n"
        "TAGS:"
    )

# ======= BACKEND LLM (Ollama HTTP) =======
def ollama_generate(prompt: str, model: str, timeout_s: int = 90) -> str:
    """
    Chamada simples ao endpoint /api/generate do Ollama.
    """
    url = f"{HOST}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        # Você pode colocar parâmetros do modelo aqui se quiser
        # "options": {"temperature": 0.0}
    }
    # timeout=(connect, read) → mata se passar de 90s
    resp = requests.post(url, json=payload, timeout=(10, timeout_s))
    resp.raise_for_status()
    data = resp.json()
    return data.get("response", "").strip()

# ======= PARSE DA SAÍDA =======
def parse_tags(output_text: str) -> List[str]:
    """
    Espera algo como: "B-PER I-PER O B-LOC ..."
    """
    # remove prefixos acidentais (ex: "TAGS:"), limpa formatação
    out = output_text.replace("\n", " ").replace("TAGS:", "").strip()
    parts = out.split()
    return parts

# ======= SALVAR/RETOMAR =======
def sha1_of_tokens(tokens: List[str]) -> str:
    msg = " ".join(tokens)
    return hashlib.sha1(msg.encode("utf-8")).hexdigest()

def load_jsonl(path: str) -> List[dict]:
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def append_jsonl(path: str, record: dict) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# ======= AMOSTRAGEM RESILIENTE =======
def sample_until_successes(
    candidates: Iterable[Dict],
    few_shots: List[Dict],
    target_successes: int = 500,
    model_key: str = MODEL_KEY,
    split_name: str = "std",
    timeout_s: int = 90,
    seed: int = 42,
) -> Tuple[int, int]:
    """
    Percorre 'candidates' na ordem (ou embaralhado, se quiser),
    tentando obter 'target_successes' sucessos.
    - Se falhar (erro/timeout), pula e tenta o próximo.
    - Salva incrementalmente.
    Retorna (num_success, num_attempted).
    """
    random.seed(seed)

    # Arquivos de saída por split
    results_path = os.path.join(MODEL_DIR, f"{split_name}_results.jsonl")
    errors_path  = os.path.join(MODEL_DIR, f"{split_name}_errors.jsonl")
    state_path   = os.path.join(MODEL_DIR, f"{split_name}_state.json")

    # Estado prévio (para retomar)
    done_ids = set()
    if os.path.exists(state_path):
        with open(state_path, "r", encoding="utf-8") as f:
            st = json.load(f)
        done_ids = set(st.get("done_ids", []))
        successes_so_far = st.get("successes", 0)
        attempted_so_far = st.get("attempted", 0)
    else:
        successes_so_far = 0
        attempted_so_far = 0

    # Para retomar caso já tenhamos rodado parcialmente
    successes = successes_so_far
    attempted = attempted_so_far

    # Loop principal
    for item in candidates:
        # id único do item pela hash dos tokens (ou use sentence_id se tiver)
        uid = item.get("uid") or item.get("sentence_id") or sha1_of_tokens(item["tokens"])

        if uid in done_ids:
            continue  # já processado

        attempted += 1

        prompt = build_prompt(few_shots, item["tokens"])

        try:
            t0 = time.time()
            raw = ollama_generate(prompt, model=model_key, timeout_s=timeout_s)
            dt = time.time() - t0

            pred_tags = parse_tags(raw)

            # verificação rápida: mesmo comprimento de tokens vs tags
            if len(pred_tags) != len(item["tokens"]):
                raise ValueError(f"Tamanho de tags != tokens ({len(pred_tags)} vs {len(item['tokens'])})")

            rec = {
                "uid": uid,
                "tokens": item["tokens"],
                "pred_tags": pred_tags,
                "elapsed_s": round(dt, 3),
                "model": model_key,
                "split": split_name,
            }
            append_jsonl(results_path, rec)
            append_jsonl(CACHE_PATH, {"type": "result", **rec})
            successes += 1
            done_ids.add(uid)

        except (requests.Timeout, requests.ConnectionError) as e:
            # timeout de 90s (ou erro de rede) → registra e segue
            err = {
                "uid": uid,
                "error": f"timeout/conn: {str(e)}",
                "model": model_key,
                "split": split_name,
            }
            append_jsonl(errors_path, err)
            append_jsonl(CACHE_PATH, {"type": "error", **err})
            done_ids.add(uid)

        except Exception as e:
            # qualquer outro erro de parsing/consistência
            err = {
                "uid": uid,
                "error": repr(e),
                "model": model_key,
                "split": split_name,
            }
            append_jsonl(errors_path, err)
            append_jsonl(CACHE_PATH, {"type": "error", **err})
            done_ids.add(uid)

        # salva estado a cada iteração (barato e seguro)
        with open(state_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "done_ids": list(done_ids),
                    "successes": successes,
                    "attempted": attempted,
                    "target_successes": target_successes,
                    "model": model_key,
                    "split": split_name,
                    "timestamp": time.time(),
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        if successes >= target_successes:
            break

    return successes, attempted



NotADirectoryError: [WinError 267] O nome do diretório é inválido: './checkpoints\\gemma2:9b'

In [ ]:
%run "splits.ipynb"  

In [ ]:
SPLIT_NAME = "test"  # "train" | "dev"/"val" | "test"
SEED = 42

# 1) dataset do notebook -> lista simples
target_ds = split_base[SPLIT_NAME]     # precisa existir no notebook
data = dataset_to_list(target_ds)

# 2) few-shot do seu próprio helper
few = select_few_shot(split_base["train"], k=FEW_SHOT_K)
few_block = _format_few_shot_block(few)

In [ ]:
JSON_PATH = "../data/geocorpus-v2.json"        # ajuste se estiver noutra pasta
SEED_GLOBAL = 42
FEW_SHOT_K = 20

random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)

# ---------- ler o arquivo ----------
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)



In [ ]:
records_geo = [
    {
        "sentence_id": i,
        "tokens"     : item["tokens"],
        "ner_tags"   : item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [ ]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [ ]:
standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
loc = loc_split(geocorpus_full)
print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

In [ ]:
splits = [standard_split, heur_len, heur_rare, advers]

In [ ]:
splits_names = ['standard', 'heur_len', 'heur_rare', 'adversarial']

In [ ]:
# ======= PIPELINE POR SPLIT =======
def run_split(
    split_name: str,
    few_shot_examples: List[Dict],
    test_items: List[Dict],
    target_successes: int = 500,
    timeout_s: int = 90,
    shuffle_candidates: bool = False,
    seed: int = 42,
) -> Dict:
    """
    few_shot_examples: lista de dicts {"tokens":[...], "tags":[...]}
    test_items: lista de dicts {"tokens":[...]} (sem tag ouro aqui)
    """
    # embaralhar candidatos é opcional
    candidates = test_items[:]
    if shuffle_candidates:
        random.Random(seed).shuffle(candidates)

    ok, attempts = sample_until_successes(
        candidates=candidates,
        few_shots=few_shot_examples,
        target_successes=target_successes,
        model_key=MODEL_KEY,
        split_name=split_name,
        timeout_s=timeout_s,
        seed=seed,
    )
    return {
        "split": split_name,
        "model": MODEL_KEY,
        "target_successes": target_successes,
        "successes": ok,
        "attempted": attempts,
    }

# ======= EXEMPLO DE USO =======
if __name__ == "__main__":
    """
    Exemplo mínimo: suponha que você já tenha seus splits prontos.
    Cada split tem seus few-shots e seu conjunto de teste.

    Estrutura esperada:
      splits = {
        "std": {
            "few": [{"tokens":[...], "tags":[...]}, ...],
            "test": [{"tokens":[...]}, ...],
        },
        "advs": {...},
        "loc":  {...},
        ...
      }
    """
    # --------> substitua por sua carga real de dados
    splits: Dict[str, Dict[str, List[Dict]]] = {}

    # Carregue seus splits (ex: de arquivos, HuggingFace, etc.)
    # Aqui vai um mock mínimo só como referência:
    # splits["std"] = {
    #     "few":  [{"tokens": ["João", "mora", "em", "São", "Paulo"], "tags": ["B-PER", "O", "O", "B-LOC", "I-LOC"]}],
    #     "test": [{"tokens": ["Maria", "está", "no", "Rio"]},
    #              {"tokens": ["IBM", "lançou", "produto"]}, ...]
    # }

    report = []
    for split_name, data in splits.items():
        few = data.get("few", [])
        test = data.get("test", [])

        # Executa com timeout=90s por item, colhendo até 500 sucessos
        stats = run_split(
            split_name=split_name,
            few_shot_examples=few,
            test_items=test,
            target_successes=500,
            timeout_s=90,
            shuffle_candidates=False,  # mude para True se quiser espalhar melhor
            seed=42,
        )
        print(stats)
        report.append(stats)

    # Salva um sumário geral do run
    summary_path = os.path.join(MODEL_DIR, "summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)